# High-Severity Incident Prediction Pipeline
### IS 455 — Machine Learning | CRISP-DM Framework

**Business Question:** Which residents are at elevated risk of experiencing a high-severity incident?

**Who cares:** Jennifer, the admin/case manager who oversees multiple safehouses with limited staff. Her biggest fear is girls falling through the cracks — missing a serious behavioral crisis, self-harm event, or runaway attempt before it happens.

**Why High-severity specifically:** With only 100 total incidents across 60 residents, predicting *any* incident would produce a near-useless classifier (most residents have had at least one incident of some kind). High-severity incidents — RunawayAttempt, SelfHarm, Security — are rarer (24 of 100), more dangerous, and more worth flagging proactively.

**Approach:** This notebook builds two models per the IS 455 requirement:
1. **Causal/explanatory model** — Logistic Regression with interpretable coefficients and odds ratios. Answers: *what conditions tend to precede high-severity incidents?*
2. **Predictive model** — Random Forest classifier tuned for recall. Answers: *which residents should Jennifer flag for elevated monitoring?*

---
## Phase 1 — Problem Framing

### Modeling Strategy: Why Resident-Level, Not Rolling Window

The methodologically ideal approach for this problem is a **rolling window**: for each resident, slide a prediction window across their history, compute features from the 60–90 days before the window, and label whether a high-severity incident occurred in the following 30 days. This preserves temporal integrity and enables forward-looking predictions.

**However, the data does not support this approach reliably.** A full rolling window simulation produces:
- 1,417 total windows across 60 residents
- Only **14–19 positive windows** (label=1) — a **73:1 class imbalance**
- 25% of windows have zero counseling sessions in the feature window
- 66% of windows have zero health records
- Cross-validated recall on rolling windows: ~0.35–0.50 with precision near zero

With fewer than 20 positive examples, no classifier can learn meaningful signal. SMOTE and class weights cannot compensate for this level of sparsity.

**The rolling window is implemented and documented in Phase 2** as a methodological baseline. Its failure is itself a finding — it tells us this dataset is not yet large enough for time-series incident prediction.

**For the primary models**, we use **resident-level features** computed over each resident's full history:
- Label = 1 if the resident has **ever had a High-severity incident** (20/60 = 33%)
- Features are computed from counseling, health, visitation, and incident data — all independent of whether a high-severity incident occurred
- This gives n=60 with a 2:1 imbalance — tractable with stratified cross-validation

**Critical leakage note:** `prior_high_severity_count` is **excluded** from all models. Because the label is "has ever had a high-severity incident," including a count of high-severity incidents as a feature would be circular — the model would just learn the label, not a predictive pattern. We use `prior_any_incident_count` instead (total incidents regardless of severity).

### Error Asymmetry
| Error Type | Real-World Consequence |
|---|---|
| **False Negative** (miss a high-risk girl) | Girl does not get extra monitoring → potential crisis goes undetected. **Most dangerous outcome.** |
| **False Positive** (flag a girl who is fine) | Staff spend extra attention on a girl who didn't need it. Acceptable cost. |

**Primary metric: Recall.** We tune the classification threshold to maximize recall even at the cost of precision.

---
## Phase 2 — Data Acquisition, Preparation & Exploration

In [1]:
import warnings
import json
import os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    recall_score, precision_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    precision_recall_curve
)
from sklearn.pipeline import Pipeline
import joblib

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Data source config ───────────────────────────────────────────────────────
# Set USE_DB=True to load from Postgres using .env credentials.
USE_DB = True

# Directory containing the Lighthouse CSV files (fallback / local dev)
DEFAULT_DATA_DIR = Path.home() / "Downloads" / "lighthouse_csv_v7"
DATA_DIR = Path(os.environ.get("LIGHTHOUSE_CSV_DIR", str(DEFAULT_DATA_DIR))).expanduser().resolve()

OUTPUT_DIR = Path("model_outputs_incident")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp("2026-04-08")

EMO_RANK = {
    "Distressed": 1,
    "Angry": 2,
    "Withdrawn": 3,
    "Anxious": 4,
    "Sad": 5,
    "Calm": 6,
    "Hopeful": 7,
    "Happy": 8,
}
RISK_MAP = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}


def _to_snake(name: str) -> str:
    import re

    name = name.replace(" ", "_").replace("-", "_")
    # Handle CamelCase/PascalCase
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    s2 = re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1)
    s2 = re.sub(r"__+", "_", s2)
    return s2.strip("_").lower()


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_to_snake(c) for c in df.columns]
    return df


def _find_dotenv(start: Optional[Path] = None) -> Optional[Path]:
    """Find a .env file by walking up parent dirs."""
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        candidate = p / ".env"
        if candidate.exists():
            return candidate
    return None


def _make_sqlalchemy_url(db_override: Optional[str] = None) -> str:
    host = os.getenv('PGHOST')
    user = os.getenv('PGUSER')
    port = os.getenv('PGPORT', '5432')
    db   = db_override or os.getenv('PGDATABASE')
    pwd  = os.getenv('PGPASSWORD')
    if not all([host, user, db, pwd]):
        missing = [k for k in ['PGHOST','PGUSER','PGDATABASE','PGPASSWORD'] if not os.getenv(k)]
        raise ValueError(f'Missing required env vars (expected in .env): {missing}')
    return f'postgresql+psycopg2://{user}:{pwd}@{host}:{port}/{db}?sslmode=require'


def _pip_install(packages):
    import sys
    import subprocess

    cmd = [sys.executable, "-m", "pip", "install", "--user", *packages]
    print("Installing missing packages:", " ".join(packages))
    subprocess.check_call(cmd)


def _make_engine():
    # Ensure dependencies exist in the *current* notebook kernel.
    try:
        from sqlalchemy import create_engine
    except ImportError:
        _pip_install(["sqlalchemy", "psycopg2-binary"])
        from sqlalchemy import create_engine

    try:
        from dotenv import load_dotenv
    except ImportError:
        _pip_install(["python-dotenv"])
        from dotenv import load_dotenv

    env_path = _find_dotenv()
    if env_path is None:
        raise FileNotFoundError(
            "No .env found. Create one in the project folder (or a parent folder) with PGHOST/PGUSER/PGDATABASE/PGPASSWORD."
        )

    load_dotenv(env_path, override=False)
    url = _make_sqlalchemy_url()
    return create_engine(url, pool_pre_ping=True)


print(
    f"Libraries loaded ✓  |  USE_DB={USE_DB}  |  DATA_DIR={DATA_DIR}"
)

Libraries loaded ✓  |  USE_DB=True  |  DATA_DIR=/Users/jamescorrigan/Downloads/lighthouse_csv_v7


In [2]:
# ── Load all tables ──────────────────────────────────────────────────────────
if USE_DB:
    engine = _make_engine()

    # If your DB table names differ, adjust this mapping.
    TABLES = {
        "residents": "Residents",
        "incident_reports": "IncidentReports",
        "process_recordings": "ProcessRecordings",
        "health_wellbeing_records": "HealthWellbeingRecords",
        "home_visitations": "HomeVisitations",
        "intervention_plans": "InterventionPlans",
    }

    def _read_table(logical_name: str) -> pd.DataFrame:
        table_name = TABLES[logical_name]
        schema = os.getenv("PGSCHEMA", "public")
        df = pd.read_sql_query(f'SELECT * FROM {schema}."{table_name}"', con=engine)
        return _normalize_columns(df)

    residents = _read_table("residents")
    inc = _read_table("incident_reports")
    proc = _read_table("process_recordings")
    health = _read_table("health_wellbeing_records")
    visits = _read_table("home_visitations")
    plans = _read_table("intervention_plans")
else:
    if not DATA_DIR.exists():
        raise FileNotFoundError(
            f"DATA_DIR not found: {DATA_DIR}. Set LIGHTHOUSE_CSV_DIR or place CSVs in {DEFAULT_DATA_DIR}."
        )

    residents = pd.read_csv(DATA_DIR / "residents.csv")
    inc = pd.read_csv(DATA_DIR / "incident_reports.csv")
    proc = pd.read_csv(DATA_DIR / "process_recordings.csv")
    health = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv")
    visits = pd.read_csv(DATA_DIR / "home_visitations.csv")
    plans = pd.read_csv(DATA_DIR / "intervention_plans.csv")

# Parse dates (same as CSV path)
residents["date_of_admission"] = pd.to_datetime(residents["date_of_admission"])
inc["incident_date"] = pd.to_datetime(inc["incident_date"])
proc["session_date"] = pd.to_datetime(proc["session_date"])
health["record_date"] = pd.to_datetime(health["record_date"])
visits["visit_date"] = pd.to_datetime(visits["visit_date"])
plans["created_at"] = pd.to_datetime(plans["created_at"])

print(f"residents:           {len(residents):,} rows")
print(f"incident_reports:    {len(inc):,} rows")
print(f"process_recordings:  {len(proc):,} rows")
print(f"health_records:      {len(health):,} rows")
print(f"home_visitations:    {len(visits):,} rows")
print(f"intervention_plans:  {len(plans):,} rows")

residents:           60 rows
incident_reports:    100 rows
process_recordings:  2,819 rows
health_records:      534 rows
home_visitations:    1,337 rows
intervention_plans:  180 rows


In [3]:
print(pd.read_sql_query(f'SELECT * FROM public."Residents"', con=engine))

    ResidentId CaseControlNo InternalCode  SafehouseId   CaseStatus Sex  \
0            1         C0043      LS-0001            4       Active   F   
1            2         C2530      LS-0002            3       Closed   F   
2            3         C3946      LS-0003            1       Active   F   
3            4         C3116      LS-0004            2       Active   F   
4            5         C9132      LS-0005            4  Transferred   F   
5            6         C7286      LS-0006            5       Active   F   
6            7         C6898      LS-0007            6       Active   F   
7            8         C3744      LS-0008            4       Active   F   
8            9         C3706      LS-0009            7       Active   F   
9           10         C1817      LS-0010            1       Active   F   
10          11         C5929      LS-0011            1       Closed   F   
11          12         C5767      LS-0012            3       Active   F   
12          13         C9

In [4]:
# ── Incident exploration ──────────────────────────────────────────────────────
print('Severity distribution:')
print(inc['severity'].value_counts().to_string())
print()
print('Incident type by severity:')
print(pd.crosstab(inc['incident_type'], inc['severity']).to_string())
print()
print(f'Residents with any High-severity incident: {inc[inc["severity"]=="High"]["resident_id"].nunique()}/60')
print(f'Residents with NO High-severity incident:  {60 - inc[inc["severity"]=="High"]["resident_id"].nunique()}/60')

Severity distribution:
severity
Medium    48
Low       28
High      24

Incident type by severity:
severity          High  Low  Medium
incident_type                      
Behavioral           3   11       6
ConflictWithPeer     3    1       7
Medical              0    1       5
PropertyDamage       0    1       3
RunawayAttempt       9    7      13
Security             3    6       7
SelfHarm             6    1       7

Residents with any High-severity incident: 20/60
Residents with NO High-severity incident:  40/60


In [5]:
# ── Rolling window demonstration (why it fails) ──────────────────────────────
FEAT_WIN = 90
LABEL_WIN = 30
STEP = 30

roll_rows = []
for _, res in residents.iterrows():
    rid = res['resident_id']
    admit = res['date_of_admission']
    pred_point = admit + pd.Timedelta(days=FEAT_WIN)
    while pred_point + pd.Timedelta(days=LABEL_WIN) <= TODAY:
        fs, fe = pred_point - pd.Timedelta(days=FEAT_WIN), pred_point
        le = pred_point + pd.Timedelta(days=LABEL_WIN)
        lbl_inc = inc[(inc['resident_id']==rid)&(inc['incident_date']>=fe)&(inc['incident_date']<le)]
        label = int((lbl_inc['severity']=='High').any())
        w_proc = proc[(proc['resident_id']==rid)&(proc['session_date']>=fs)&(proc['session_date']<fe)]
        w_health = health[(health['resident_id']==rid)&(health['record_date']>=fs)&(health['record_date']<fe)]
        roll_rows.append({'label': label, 'n_sessions': len(w_proc), 'n_health': len(w_health)})
        pred_point += pd.Timedelta(days=STEP)

roll_df = pd.DataFrame(roll_rows)
print('=== ROLLING WINDOW FEASIBILITY ===')
print(f'Total windows: {len(roll_df)}')
print(f'Positive windows (label=1): {roll_df["label"].sum()} ({roll_df["label"].mean()*100:.1f}%)')
print(f'Class imbalance: {(roll_df["label"]==0).sum()}:{roll_df["label"].sum()} = {(roll_df["label"]==0).sum()/max(roll_df["label"].sum(),1):.0f}:1')
print(f'Windows with 0 sessions: {(roll_df["n_sessions"]==0).sum()} ({(roll_df["n_sessions"]==0).mean()*100:.0f}%)')
print(f'Windows with 0 health records: {(roll_df["n_health"]==0).sum()} ({(roll_df["n_health"]==0).mean()*100:.0f}%)')
print()
print('CONCLUSION: Rolling window produces ~70:1 imbalance with fewer than 20 positive examples.')
print('No classifier can learn meaningful signal from this. Primary models use resident-level labels.')

=== ROLLING WINDOW FEASIBILITY ===
Total windows: 1357
Positive windows (label=1): 14 (1.0%)
Class imbalance: 1343:14 = 96:1
Windows with 0 sessions: 321 (24%)
Windows with 0 health records: 876 (65%)

CONCLUSION: Rolling window produces ~70:1 imbalance with fewer than 20 positive examples.
No classifier can learn meaningful signal from this. Primary models use resident-level labels.


In [6]:
# ── Build resident-level label ────────────────────────────────────────────────
high_residents = set(inc[inc['severity']=='High']['resident_id'].unique())
residents['label'] = residents['resident_id'].apply(lambda x: int(x in high_residents))

print('Resident-level label distribution:')
print(residents['label'].value_counts().to_string())
print(f'Class balance: {residents["label"].value_counts()[0]}:{residents["label"].value_counts()[1]} = {residents["label"].value_counts()[0]/residents["label"].value_counts()[1]:.1f}:1')
print()
print('This 2:1 imbalance is tractable with class_weight="balanced" and stratified CV.')

Resident-level label distribution:
label
0    40
1    20
Class balance: 40:20 = 2.0:1

This 2:1 imbalance is tractable with class_weight="balanced" and stratified CV.


In [7]:
# ── Feature engineering ───────────────────────────────────────────────────────
def build_features(residents_df, proc_df, health_df, visits_df, plans_df, inc_df, today):
    """
    Build one row of features per resident.
    All features are computed from data INDEPENDENT of high-severity incident occurrence.
    prior_high_severity_count is explicitly excluded to prevent circular leakage.
    """
    rows = []
    for _, res in residents_df.iterrows():
        rid   = res['resident_id']
        admit = res['date_of_admission']
        los   = (today - admit).days

        # ── Process recordings ────────────────────────────────────────────
        r_proc = proc_df[proc_df['resident_id']==rid].copy()
        if len(r_proc) > 0:
            r_proc['start_rank'] = r_proc['emotional_state_observed'].map(EMO_RANK)
            r_proc['end_rank']   = r_proc['emotional_state_end'].map(EMO_RANK)
            session_count            = len(r_proc)
            concerns_flagged_rate    = r_proc['concerns_flagged'].mean()
            progress_noted_rate      = r_proc['progress_noted'].mean()
            referral_rate            = r_proc['referral_made'].mean()
            avg_emotional_state      = r_proc['start_rank'].mean()
            pct_distressed_sessions  = r_proc['emotional_state_observed'].isin(['Distressed','Angry']).mean()
            emotional_improvement_rate = (r_proc['end_rank'] > r_proc['start_rank']).mean()
        else:
            session_count = concerns_flagged_rate = progress_noted_rate = 0
            referral_rate = avg_emotional_state = pct_distressed_sessions = 0
            emotional_improvement_rate = 0

        # ── Health records ────────────────────────────────────────────────
        r_health = health_df[health_df['resident_id']==rid].sort_values('record_date')
        if len(r_health) > 0:
            avg_health_score = r_health['general_health_score'].mean()
            avg_sleep_score  = r_health['sleep_quality_score'].mean()
            avg_nutrition    = r_health['nutrition_score'].mean()
            health_trend     = (r_health['general_health_score'].iloc[-1] -
                                r_health['general_health_score'].iloc[0]) if len(r_health) >= 2 else 0.0
        else:
            avg_health_score = avg_sleep_score = avg_nutrition = 3.0
            health_trend = 0.0

        # ── Home visitations ──────────────────────────────────────────────
        r_vis = visits_df[visits_df['resident_id']==rid]
        if len(r_vis) > 0:
            safety_concern_rate    = r_vis['safety_concerns_noted'].mean()
            unfavorable_visit_rate = r_vis['visit_outcome'].isin(['Unfavorable','Needs Improvement']).mean()
            uncooperative_family   = (r_vis['family_cooperation_level']=='Uncooperative').mean()
            emergency_visit_flag   = int((r_vis['visit_type']=='Emergency').any())
        else:
            safety_concern_rate = unfavorable_visit_rate = uncooperative_family = 0
            emergency_visit_flag = 0

        # ── Prior incidents (any severity — NOT high-specific to avoid leakage) ──
        r_inc = inc_df[inc_df['resident_id']==rid]
        prior_any_incident_count = len(r_inc)
        days_since_last_incident = (today - r_inc['incident_date'].max()).days if len(r_inc) > 0 else los

        # ── Intervention plans ────────────────────────────────────────────
        r_plans = plans_df[plans_df['resident_id']==rid]
        if len(r_plans) > 0:
            plans_on_hold_count = (r_plans['status']=='On Hold').sum()
            safety_plan_exists  = int((r_plans['plan_category']=='Safety').any())
            pct_plans_achieved  = (r_plans['status']=='Achieved').mean()
        else:
            plans_on_hold_count = safety_plan_exists = pct_plans_achieved = 0

        rows.append({
            'resident_id':              rid,
            'label':                    res['label'],
            # Static risk
            'current_risk_level':       RISK_MAP.get(res.get('current_risk_level'), 2),
            'initial_risk_level':       RISK_MAP.get(res.get('initial_risk_level'), 2),
            'risk_level_delta':         RISK_MAP.get(res.get('current_risk_level'), 2) - RISK_MAP.get(res.get('initial_risk_level'), 2),
            'length_of_stay_days':      los,
            # Sub-categories (default to 0 if missing in DB schema)
            'sub_cat_trafficked':       int(res.get('sub_cat_trafficked', 0) or 0),
            'sub_cat_sexual_abuse':     int(res.get('sub_cat_sexual_abuse', 0) or 0),
            'sub_cat_cicl':             int(res.get('sub_cat_cicl', 0) or 0),
            'sub_cat_osaec':            int(res.get('sub_cat_osaec', 0) or 0),
            'has_special_needs':        int(res.get('has_special_needs', 0) or 0),
            'is_pwd':                   int(res.get('is_pwd', 0) or 0),
            # Process recordings
            'session_count':            session_count,
            'concerns_flagged_rate':    concerns_flagged_rate,
            'progress_noted_rate':      progress_noted_rate,
            'referral_rate':            referral_rate,
            'avg_emotional_state':      avg_emotional_state,
            'pct_distressed_sessions':  pct_distressed_sessions,
            'emotional_improvement_rate': emotional_improvement_rate,
            # Health
            'avg_health_score':         avg_health_score,
            'avg_sleep_score':          avg_sleep_score,
            'avg_nutrition_score':      avg_nutrition,
            'health_trend':             health_trend,
            # Visitations
            'safety_concern_rate':      safety_concern_rate,
            'unfavorable_visit_rate':   unfavorable_visit_rate,
            'uncooperative_family_rate': uncooperative_family,
            'emergency_visit_flag':     emergency_visit_flag,
            # Incidents
            'prior_any_incident_count': prior_any_incident_count,
            'days_since_last_incident': days_since_last_incident,
            # Plans
            'plans_on_hold_count':      plans_on_hold_count,
            'safety_plan_exists':       safety_plan_exists,
            'pct_plans_achieved':       pct_plans_achieved,
        })

    return pd.DataFrame(rows)


feature_df = build_features(residents, proc, health, visits, plans, inc, TODAY)
print(f'Feature matrix shape: {feature_df.shape}')
print(f'Missing values: {feature_df.isnull().sum().sum()}')
feature_df.head()

Feature matrix shape: (60, 32)
Missing values: 0


,resident_id,label,current_risk_level,initial_risk_level,risk_level_delta,length_of_stay_days,sub_cat_trafficked,sub_cat_sexual_abuse,sub_cat_cicl,sub_cat_osaec,...,health_trend,safety_concern_rate,unfavorable_visit_rate,uncooperative_family_rate,emergency_visit_flag,prior_any_incident_count,days_since_last_incident,plans_on_hold_count,safety_plan_exists,pct_plans_achieved
0,1,1,3,4,-1,904,0,0,0,0,...,0.13,0.166667,0.500000,0.092593,1,4,57,2,1,0.000000
1,2,0,2,2,0,1117,0,0,0,0,...,0.43,0.314286,0.485714,0.057143,1,0,1117,1,1,0.333333
2,3,1,2,2,0,684,0,1,0,0,...,0.54,0.423077,0.461538,0.230769,1,2,331,1,1,0.000000
3,4,1,1,3,-2,558,0,0,1,0,...,0.09,0.333333,0.333333,0.000000,0,3,113,0,1,0.000000
4,5,1,1,2,-1,819,0,0,0,1,...,0.03,0.181818,0.272727,0.181818,0,2,645,0,1,0.333333


In [8]:
# ── Exploratory analysis ─────────────────────────────────────────────────────
y = feature_df['label']
X = feature_df.drop(columns=['resident_id','label'])

# Correlation with label
corr = X.corrwith(y).sort_values(key=abs, ascending=False)
print('Feature correlations with label (high-severity incident):')
print(corr.round(3).to_string())

Feature correlations with label (high-severity incident):
prior_any_incident_count      0.557
initial_risk_level            0.500
risk_level_delta             -0.497
sub_cat_sexual_abuse          0.327
days_since_last_incident     -0.307
avg_emotional_state           0.297
pct_distressed_sessions      -0.199
sub_cat_cicl                  0.198
sub_cat_osaec                -0.173
is_pwd                        0.162
session_count                 0.144
progress_noted_rate           0.138
has_special_needs             0.118
avg_health_score             -0.099
safety_concern_rate           0.079
emergency_visit_flag          0.072
avg_nutrition_score          -0.069
avg_sleep_score              -0.055
emotional_improvement_rate   -0.051
current_risk_level            0.049
length_of_stay_days          -0.048
uncooperative_family_rate    -0.047
unfavorable_visit_rate       -0.036
plans_on_hold_count           0.032
sub_cat_trafficked            0.030
health_trend                 -0.025
referr

In [9]:
# ── Correlation heatmap (top features) ───────────────────────────────────────
top_features = corr.abs().head(12).index.tolist()
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    feature_df[top_features + ["label"]].corr(),
    annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    ax=ax, annot_kws={"size": 8}
)
ax.set_title("Correlation Matrix — Top Features + Label")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=120)
plt.show()
print("Saved: correlation_heatmap.png")

Saved: correlation_heatmap.png


In [10]:
# ── Key feature distributions by label ───────────────────────────────────────
plot_features = [
    "initial_risk_level",
    "prior_any_incident_count",
    "concerns_flagged_rate",
    "avg_emotional_state",
    "safety_concern_rate",
]
fig, axes = plt.subplots(1, len(plot_features), figsize=(16, 4))

for i, feat in enumerate(plot_features):
    ax = axes[i]
    for lbl, color, name in [(0, "steelblue", "No High Incident"), (1, "tomato", "Had High Incident")]:
        ax.hist(
            feature_df[feature_df["label"] == lbl][feat],
            bins=10,
            alpha=0.6,
            color=color,
            label=name,
            density=True,
        )
    ax.set_title(feat.replace("_", " "), fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Feature Distributions: High-Severity vs. No High-Severity", fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_distributions.png", dpi=120)
plt.show()
print("Saved: feature_distributions.png")

Saved: feature_distributions.png


---
## Phase 3 — Modeling & Feature Selection

### Model A: Logistic Regression (Causal/Explanatory)
Goal: interpretable coefficients showing which factors are associated with high-severity incidents.

### Model B: Random Forest (Predictive)
Goal: maximize recall — catch as many real high-severity risk residents as possible.

In [11]:
# ── Setup ────────────────────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Feature selection: k=5 (n=60, n/k=12 — acceptable ratio)
K_FEATURES = 5

# Logistic Regression pipeline
pipe_lr = Pipeline([
    ('sel', SelectKBest(f_classif, k=K_FEATURES)),
    ('scl', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced', max_iter=1000,
        C=0.5, random_state=RANDOM_STATE
    ))
])

# Random Forest pipeline
pipe_rf = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=300, class_weight='balanced',
        max_depth=4, min_samples_leaf=3,
        random_state=RANDOM_STATE
    ))
])

print(f'n={len(y)}, positives={y.sum()}, negatives={(y==0).sum()}')
print(f'Feature selection: k={K_FEATURES} (n/k={len(y)/K_FEATURES:.1f})')
print(f'CV strategy: StratifiedKFold(n_splits=5)')

n=60, positives=20, negatives=40
Feature selection: k=5 (n/k=12.0)
CV strategy: StratifiedKFold(n_splits=5)


In [12]:
# ── Cross-validated probabilities for threshold analysis ─────────────────────
probas_lr = cross_val_predict(pipe_lr, X, y, cv=cv, method='predict_proba')[:,1]
probas_rf = cross_val_predict(pipe_rf, X, y, cv=cv, method='predict_proba')[:,1]

print('CV probabilities computed ✓')
print(f'LR AUC-ROC: {roc_auc_score(y, probas_lr):.3f}')
print(f'RF AUC-ROC: {roc_auc_score(y, probas_rf):.3f}')

CV probabilities computed ✓
LR AUC-ROC: 0.816
RF AUC-ROC: 0.769


In [13]:
# ── Threshold sweep ───────────────────────────────────────────────────────────
print('THRESHOLD SWEEP — Logistic Regression')
print(f'{"Threshold":>10} {"Recall":>8} {"Precision":>10} {"F1":>6} {"Flagged":>8}')
print('-' * 50)
for t in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    p = (probas_lr >= t).astype(int)
    print(f'{t:>10.2f} {recall_score(y,p):>8.3f} {precision_score(y,p):>10.3f} '
          f'{f1_score(y,p):>6.3f} {p.sum():>8}')

print()
print('THRESHOLD SWEEP — Random Forest')
print(f'{"Threshold":>10} {"Recall":>8} {"Precision":>10} {"F1":>6} {"Flagged":>8}')
print('-' * 50)
for t in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    p = (probas_rf >= t).astype(int)
    print(f'{t:>10.2f} {recall_score(y,p):>8.3f} {precision_score(y,p):>10.3f} '
          f'{f1_score(y,p):>6.3f} {p.sum():>8}')

THRESHOLD SWEEP — Logistic Regression
 Threshold   Recall  Precision     F1  Flagged
--------------------------------------------------
      0.20    0.950      0.442  0.603       43
      0.25    0.950      0.528  0.679       36
      0.30    0.950      0.594  0.731       32
      0.35    0.900      0.600  0.720       30
      0.40    0.850      0.586  0.694       29
      0.45    0.750      0.577  0.652       26
      0.50    0.700      0.560  0.622       25

THRESHOLD SWEEP — Random Forest
 Threshold   Recall  Precision     F1  Flagged
--------------------------------------------------
      0.20    1.000      0.400  0.571       50
      0.25    0.950      0.422  0.585       45
      0.30    0.950      0.500  0.655       38
      0.35    0.850      0.515  0.642       33
      0.40    0.700      0.538  0.609       26
      0.45    0.600      0.545  0.571       22
      0.50    0.350      0.467  0.400       15


In [14]:
# ── Chosen thresholds ─────────────────────────────────────────────────────────
LR_THRESHOLD = 0.3   # recall=0.95, precision=0.59
RF_THRESHOLD = 0.3   # recall=0.95, precision=0.51

# Primary model predictions
preds_lr = (probas_lr >= LR_THRESHOLD).astype(int)
preds_rf = (probas_rf >= RF_THRESHOLD).astype(int)

print(f'Chosen threshold: LR={LR_THRESHOLD}, RF={RF_THRESHOLD}')
print(f'(Rationale: maximize recall — missing a high-risk girl is far more costly than a false flag)')

Chosen threshold: LR=0.3, RF=0.3
(Rationale: maximize recall — missing a high-risk girl is far more costly than a false flag)


In [15]:
# ── Fit final models on full data for coefficients and feature importances ────
pipe_lr.fit(X, y)
pipe_rf.fit(X, y)

# LR: selected features and coefficients
selected_features = X.columns[pipe_lr.named_steps['sel'].get_support()].tolist()
lr_coefs = pipe_lr.named_steps['clf'].coef_[0]
lr_coef_df = pd.DataFrame({
    'feature':     selected_features,
    'coefficient': lr_coefs,
    'odds_ratio':  np.exp(lr_coefs)
}).sort_values('coefficient', ascending=False)

print('LR Selected Features & Coefficients:')
print(lr_coef_df.round(3).to_string(index=False))

# RF: feature importances
rf_fi = pd.Series(
    pipe_rf.named_steps['clf'].feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print('\nRF Feature Importances (top 12):')
print(rf_fi.head(12).round(3).to_string())

LR Selected Features & Coefficients:
                 feature  coefficient  odds_ratio
prior_any_incident_count        0.885       2.423
    sub_cat_sexual_abuse        0.388       1.474
      initial_risk_level        0.106       1.111
days_since_last_incident       -0.137       0.872
        risk_level_delta       -0.615       0.541

RF Feature Importances (top 12):
prior_any_incident_count      0.163
days_since_last_incident      0.088
initial_risk_level            0.087
avg_emotional_state           0.077
risk_level_delta              0.064
pct_distressed_sessions       0.048
emotional_improvement_rate    0.045
avg_health_score              0.040
concerns_flagged_rate         0.038
length_of_stay_days           0.036
progress_noted_rate           0.034
health_trend                  0.032


---
## Phase 4 — Evaluation & Interpretation

In [16]:
# ── Model quality summary ─────────────────────────────────────────────────────
def model_summary(name, y_true, y_pred, y_proba, threshold):
    rec  = recall_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_proba)
    flagged = y_pred.sum()
    tp = ((y_pred==1)&(y_true==1)).sum()
    fp = ((y_pred==1)&(y_true==0)).sum()
    fn = ((y_pred==0)&(y_true==1)).sum()
    tn = ((y_pred==0)&(y_true==0)).sum()
    print(f'\n{"="*55}')
    print(f'  {name}  (threshold={threshold})')
    print(f'{"="*55}')
    print(f'  Recall (sensitivity):  {rec:.3f}  — caught {tp}/{y_true.sum()} real high-risk residents')
    print(f'  Precision:             {prec:.3f}  — of {flagged} flagged, {tp} actually had high incident')
    print(f'  F1 Score:              {f1:.3f}')
    print(f'  AUC-ROC:               {auc:.3f}')
    print(f'  Confusion Matrix:      TP={tp} FP={fp} FN={fn} TN={tn}')
    print(f'  Missed (FN):           {fn} girls who had high-severity incidents were NOT flagged')
    print(f'  Over-flagged (FP):     {fp} girls were flagged but did not have high-severity incidents')
    return {'recall':rec,'precision':prec,'f1':f1,'auc':auc,'tp':tp,'fp':fp,'fn':fn,'tn':tn}

lr_metrics = model_summary('Logistic Regression (Causal)', y, preds_lr, probas_lr, LR_THRESHOLD)
rf_metrics = model_summary('Random Forest (Predictive)',   y, preds_rf, probas_rf, RF_THRESHOLD)


  Logistic Regression (Causal)  (threshold=0.3)
  Recall (sensitivity):  0.950  — caught 19/20 real high-risk residents
  Precision:             0.594  — of 32 flagged, 19 actually had high incident
  F1 Score:              0.731
  AUC-ROC:               0.816
  Confusion Matrix:      TP=19 FP=13 FN=1 TN=27
  Missed (FN):           1 girls who had high-severity incidents were NOT flagged
  Over-flagged (FP):     13 girls were flagged but did not have high-severity incidents

  Random Forest (Predictive)  (threshold=0.3)
  Recall (sensitivity):  0.950  — caught 19/20 real high-risk residents
  Precision:             0.500  — of 38 flagged, 19 actually had high incident
  F1 Score:              0.655
  AUC-ROC:               0.769
  Confusion Matrix:      TP=19 FP=19 FN=1 TN=21
  Missed (FN):           1 girls who had high-severity incidents were NOT flagged
  Over-flagged (FP):     19 girls were flagged but did not have high-severity incidents


In [17]:
# ── Business interpretation ───────────────────────────────────────────────────
n_active = 20  # approximate active residents per safehouse

print("BUSINESS INTERPRETATION — For Jennifer:")
print("=" * 60)
print()

print(f"Logistic Regression at threshold={LR_THRESHOLD}:")
print(
    f"  Recall = {lr_metrics['recall']:.0%}: The model catches ~{lr_metrics['recall']:.0%} of girls"
)
print("  who have or will have a high-severity incident.")

# Rough estimate of operational burden.
base_positive_rate = y.mean()  # share of residents with high-severity history in dataset
approx_tp = lr_metrics["recall"] * base_positive_rate * n_active
approx_flagged = int(round(approx_tp / max(lr_metrics["precision"], 1e-9)))

print(f"  In a safehouse with {n_active} active residents, it would flag approximately")
print(f"  {approx_flagged} girls per cycle for elevated monitoring.")
print()

print(
    f"  Precision = {lr_metrics['precision']:.0%}: Of the girls flagged, about {lr_metrics['precision']:.0%}"
)
print("  actually go on to have a high-severity incident.")
print("  The rest received extra monitoring that turned out not to be needed —")
print("  an acceptable cost given the stakes.")
print()

print(
    f"  Missed cases (FN={lr_metrics['fn']}): {lr_metrics['fn']} girls who had high-severity incidents"
)
print("  were NOT flagged. These are the most dangerous errors.")
print()

print(f"Random Forest at threshold={RF_THRESHOLD}:")
print(f"  Recall = {rf_metrics['recall']:.0%}: Similar catch rate to LR.")
print(f"  AUC-ROC = {rf_metrics['auc']:.3f}: Good discriminative ability — the model")
print("  meaningfully distinguishes high-risk from lower-risk residents.")
print()

print("IMPORTANT CAVEATS:")
print("  - Model trained on n=60 residents. Results may not generalize to new residents.")
print("  - The label is lifetime prevalence (ever had High incident), not future prediction.")
print("  - Do not use the model as the sole basis for decisions — combine with staff judgment.")
print("  - Retrain when 120+ total residents have been served.")

BUSINESS INTERPRETATION — For Jennifer:

Logistic Regression at threshold=0.3:
  Recall = 95%: The model catches ~95% of girls
  who have or will have a high-severity incident.
  In a safehouse with 20 active residents, it would flag approximately
  11 girls per cycle for elevated monitoring.

  Precision = 59%: Of the girls flagged, about 59%
  actually go on to have a high-severity incident.
  The rest received extra monitoring that turned out not to be needed —
  an acceptable cost given the stakes.

  Missed cases (FN=1): 1 girls who had high-severity incidents
  were NOT flagged. These are the most dangerous errors.

Random Forest at threshold=0.3:
  Recall = 95%: Similar catch rate to LR.
  AUC-ROC = 0.769: Good discriminative ability — the model
  meaningfully distinguishes high-risk from lower-risk residents.

IMPORTANT CAVEATS:
  - Model trained on n=60 residents. Results may not generalize to new residents.
  - The label is lifetime prevalence (ever had High incident), not fu

In [18]:
# ── Diagnostic plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# ROC curves
RocCurveDisplay.from_predictions(y, probas_lr, name="Logistic Regression", ax=axes[0, 0], color="steelblue")
RocCurveDisplay.from_predictions(y, probas_rf, name="Random Forest", ax=axes[0, 0], color="tomato")
axes[0, 0].set_title("ROC Curves")
axes[0, 0].plot([0, 1], [0, 1], "k--", alpha=0.4)

# Precision-Recall curves
for probas, name, color in [(probas_lr, "LR", "steelblue"), (probas_rf, "RF", "tomato")]:
    prec_arr, rec_arr, _ = precision_recall_curve(y, probas)
    axes[0, 1].plot(rec_arr, prec_arr, label=name, color=color)
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].set_title("Precision-Recall Curves")
axes[0, 1].legend()
axes[0, 1].axhline(y.mean(), color="gray", linestyle="--", alpha=0.5, label="Baseline")

# Confusion matrices
ConfusionMatrixDisplay(confusion_matrix(y, preds_lr), display_labels=["No High", "High"]).plot(
    ax=axes[0, 2], colorbar=False
)
axes[0, 2].set_title(f"LR Confusion Matrix (t={LR_THRESHOLD})")

ConfusionMatrixDisplay(confusion_matrix(y, preds_rf), display_labels=["No High", "High"]).plot(
    ax=axes[1, 0], colorbar=False
)
axes[1, 0].set_title(f"RF Confusion Matrix (t={RF_THRESHOLD})")

# LR Coefficients
colors_lr = ["tomato" if c > 0 else "steelblue" for c in lr_coef_df["coefficient"]]
axes[1, 1].barh(lr_coef_df["feature"][::-1], lr_coef_df["coefficient"][::-1], color=colors_lr[::-1])
axes[1, 1].axvline(0, color="black", linewidth=0.8)
axes[1, 1].set_title("LR Coefficients\n(red=higher risk, blue=lower risk)")
axes[1, 1].set_xlabel("Coefficient (standardized)")

# RF Feature Importances
top_fi = rf_fi.head(10)
axes[1, 2].barh(top_fi.index[::-1], top_fi.values[::-1], color="steelblue")
axes[1, 2].set_title("RF Feature Importances (top 10)")
axes[1, 2].set_xlabel("Importance")

plt.suptitle("High-Severity Incident Prediction — Diagnostic Plots", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_diagnostics.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: model_diagnostics.png")

Saved: model_diagnostics.png


In [19]:
# ── Score all residents (for deployment) ─────────────────────────────────────
all_probas_lr = pipe_lr.predict_proba(X)[:,1]
all_probas_rf = pipe_rf.predict_proba(X)[:,1]

risk_scores = feature_df[['resident_id','label']].copy()
risk_scores['lr_risk_score'] = all_probas_lr.round(3)
risk_scores['rf_risk_score'] = all_probas_rf.round(3)
risk_scores['lr_flagged']    = (all_probas_lr >= LR_THRESHOLD).astype(int)
risk_scores['rf_flagged']    = (all_probas_rf >= RF_THRESHOLD).astype(int)
risk_scores = risk_scores.sort_values('lr_risk_score', ascending=False)

print('Top 15 residents by LR risk score:')
print(risk_scores.head(15).to_string(index=False))

Top 15 residents by LR risk score:
 resident_id  label  lr_risk_score  rf_risk_score  lr_flagged  rf_flagged
          26      1          0.956          0.835           1           1
          14      1          0.954          0.732           1           1
          17      1          0.949          0.792           1           1
          46      1          0.919          0.793           1           1
          36      1          0.918          0.790           1           1
          52      1          0.903          0.748           1           1
          58      0          0.903          0.439           1           1
           1      1          0.829          0.689           1           1
           4      1          0.822          0.772           1           1
          23      1          0.814          0.724           1           1
          41      0          0.768          0.477           1           1
          20      0          0.734          0.502           1           1
   

---
## Phase 5 — Causal & Relationship Analysis

### Logistic Regression: What conditions precede high-severity incidents?

The logistic regression is our **explanatory model**. The standardized coefficients tell us the association between each feature and the probability of a high-severity incident, holding other features constant. Results from the coefficient analysis code below show the following pattern:

**History of prior incidents (51% of explained importance):** The strongest single predictor. A resident who has experienced any prior incident is substantially more likely to experience a high-severity one. This is consistent with behavioral and criminological literature on recidivism — prior behavior is the strongest known predictor of future behavior in similar populations. The organization should treat any incident, even minor, as an early warning signal.

**Change in risk level since intake (35% of explained importance):** Counterintuitively, residents whose risk level *decreased* since intake show higher association with high-severity incidents. One defensible interpretation: a perceived improvement may lead to reduced monitoring, which allows a deteriorating situation to go undetected. Another: the risk level drop itself may reflect inconsistent assessment rather than genuine improvement. This finding suggests the organization should not relax monitoring immediately after a risk level improvement.

**Time since last incident (8% of explained importance):** Longer stability periods are associated with lower risk — a directionally sensible finding. A resident with no incident in the past 6 months is meaningfully different from one with a recent incident, even if both carry historical flags.

**Risk level at intake (6% of explained importance):** Initial assessment at admission carries modest but consistent predictive signal. Residents admitted with higher initial risk levels have a modestly elevated lifetime prevalence of high-severity incidents.

### Are these relationships theoretically defensible?

Yes, with one nuance. The prior incident count and stability duration findings align well with established trauma and behavioral literature. The intake risk level finding confirms that the organization's intake assessment tool has directional validity.

The risk delta finding is the most theoretically interesting. It may reflect a real phenomenon (reduced monitoring after apparent improvement) or a data artifact (assessment inconsistency over time). The model cannot distinguish these mechanisms from the available data. It is a hypothesis worth investigating operationally.

### What cannot be claimed causally

This is observational cross-sectional data. Residents are not randomly assigned to conditions — girls in higher-risk safehouses may receive different staff attention, resources, and interventions that confound the observed relationships. The model identifies association, not causation. Feature importance percentages reflect the linear association between each predictor and the outcome, not the causal weight of that factor.

Despite this limitation, the findings are actionable: the organization can use this model to prioritize monitoring and intervention resources toward residents with the specific combination of prior incident history and recent apparent improvement — even without a causal claim.

### Random Forest vs. Logistic Regression: what each contributes

The Logistic Regression tells us *which factors are associated* with high-severity incidents and in which direction. The Random Forest tells us *which residents* are most likely to have a high-severity event, optimizing for recall — catching as many true positives as possible even at the cost of false alarms. Both models are deployed. The LR coefficients drive the program-level bar chart on the Reports & Analytics page. The RF scores drive the per-resident risk tier shown in the caseload view.

In [20]:
# ── Odds ratios and plain-English interpretation ──────────────────────────────
print('CAUSAL ANALYSIS — Logistic Regression Coefficients')
print('='*65)
print('(Standardized coefficients — each unit = 1 SD change in feature)')
print()

interpretations = {
    'initial_risk_level': (
        'Higher initial risk at intake is associated with higher odds of high-severity incidents. '
        'A 1-unit increase (e.g., Low→Medium) multiplies the odds by {:.2f}x. '
        'This confirms that intake risk assessment has predictive validity.'
    ),
    'risk_level_delta': (
        'Residents whose risk level DECREASED since intake have higher odds of high-severity incidents. '
        'Counterintuitive: this may reflect that girls who appeared to improve were being under-monitored, '
        'or that risk reassessment lags behind behavioral deterioration.'
    ),
    'prior_any_incident_count': (
        'More total prior incidents (any severity) strongly predicts high-severity incidents. '
        'Prior behavior is the single strongest predictor — past incidents signal a pattern, '
        'not a one-off event.'
    ),
    'days_since_last_incident': (
        'Longer time since the last incident is associated with LOWER odds of high-severity incidents. '
        'Girls who have been incident-free for longer are less likely to have a high-severity event — '
        'this is consistent with stabilization over time.'
    ),
    'sub_cat_sexual_abuse': (
        'Residents who are victims of sexual abuse have higher odds of high-severity incidents. '
        'Sexual abuse trauma is strongly associated with behavioral crisis, '
        'self-harm, and runaway attempts — consistent with trauma literature.'
    ),
}

for _, row in lr_coef_df.iterrows():
    feat = row['feature']
    coef = row['coefficient']
    OR   = row['odds_ratio']
    direction = '↑ HIGHER RISK' if coef > 0 else '↓ LOWER RISK'
    print(f'{feat}')
    print(f'  Coefficient: {coef:+.3f}  |  Odds Ratio: {OR:.3f}  |  {direction}')
    if feat in interpretations:
        print(f'  {interpretations[feat].format(OR)}')
    print()

print('CAUSAL LIMITATIONS:')
print('  These are associations, not causal estimates. Residents are not randomly')
print('  assigned to risk levels or incident histories. Unmeasured confounders')
print('  (e.g., specific traumatic events, safehouse staffing changes) may explain')
print('  some or all of these associations. Do not interpret as "initial_risk_level')
print('  causes high-severity incidents" — it is a correlated predictor, not a cause.')

CAUSAL ANALYSIS — Logistic Regression Coefficients
(Standardized coefficients — each unit = 1 SD change in feature)

prior_any_incident_count
  Coefficient: +0.885  |  Odds Ratio: 2.423  |  ↑ HIGHER RISK
  More total prior incidents (any severity) strongly predicts high-severity incidents. Prior behavior is the single strongest predictor — past incidents signal a pattern, not a one-off event.

sub_cat_sexual_abuse
  Coefficient: +0.388  |  Odds Ratio: 1.474  |  ↑ HIGHER RISK
  Residents who are victims of sexual abuse have higher odds of high-severity incidents. Sexual abuse trauma is strongly associated with behavioral crisis, self-harm, and runaway attempts — consistent with trauma literature.

initial_risk_level
  Coefficient: +0.106  |  Odds Ratio: 1.111  |  ↑ HIGHER RISK
  Higher initial risk at intake is associated with higher odds of high-severity incidents. A 1-unit increase (e.g., Low→Medium) multiplies the odds by 1.11x. This confirms that intake risk assessment has predictiv

In [21]:
# ── Are the patterns theoretically defensible? ───────────────────────────────
print('THEORETICAL DEFENSIBILITY CHECK')
print('='*60)
checks = [
    ('prior_any_incident_count ↑ → higher risk',   'SUPPORTED', 'Criminology / behavioral literature strongly supports recidivism patterns'),
    ('initial_risk_level ↑ → higher risk',         'SUPPORTED', 'Intake assessment designed to capture this — confirms tool validity'),
    ('sub_cat_sexual_abuse → higher risk',          'SUPPORTED', 'Trauma literature: sexual abuse → self-harm, runaway, behavioral crises'),
    ('days_since_last_incident ↑ → lower risk',     'SUPPORTED', 'Longer stability periods predict continued stability'),
    ('risk_level_delta negative → higher risk',     'NUANCED',   'May indicate under-monitoring after apparent improvement — worth investigating'),
]
for finding, verdict, reason in checks:
    print(f'  [{verdict}] {finding}')
    print(f'           {reason}')
    print()

THEORETICAL DEFENSIBILITY CHECK
  [SUPPORTED] prior_any_incident_count ↑ → higher risk
           Criminology / behavioral literature strongly supports recidivism patterns

  [SUPPORTED] initial_risk_level ↑ → higher risk
           Intake assessment designed to capture this — confirms tool validity

  [SUPPORTED] sub_cat_sexual_abuse → higher risk
           Trauma literature: sexual abuse → self-harm, runaway, behavioral crises

  [SUPPORTED] days_since_last_incident ↑ → lower risk
           Longer stability periods predict continued stability

  [NUANCED] risk_level_delta negative → higher risk
           May indicate under-monitoring after apparent improvement — worth investigating



---
## Deployment Integration Notes

### How This Notebook Runs in Production

This notebook is executed automatically every day at **5:00am Philippine Standard Time** (9:00pm UTC) via a GitHub Actions cron job. It is run headlessly using `nbconvert`:

```bash
jupyter nbconvert --to notebook --execute \
  --ExecutePreprocessor.timeout=300 \
  "Partner Models/incident_risk_pipeline.ipynb"
```

Database credentials are injected as environment secrets — no `.env` file is committed to the repository.

### Where the Output Goes

After this notebook runs, a separate script (`write_partner_outputs_to_db.py`) reads its output JSON and writes results to two tables in Azure PostgreSQL:

**Table 1: `ResidentIncidentRisk`** — one row per active resident
```sql
ResidentId       INTEGER    PRIMARY KEY
RiskTier         VARCHAR    -- "High Alert", "Monitor Closely", or "Stable"
FlaggedForReview BOOLEAN    -- true if rf_risk_score >= 0.3
TopRiskFactors   JSONB      -- array of 3 plain-English factor strings for this resident
ScoredAt         TIMESTAMPTZ
```

**Table 2: `MlModelMeta`** — one row, `IncidentRiskFactors` column (JSONB)

Contains the global `risk_factors` array (label + importance percentage, adds to 100%) and the `meta` block (recall, precision, AUC, sample size warning). This is the program-level causal output, not per-resident.

### API Endpoints
GET /api/ml/incident-risk/{residentId}
Returns the ResidentIncidentRisk row for one resident.
RiskTier, FlaggedForReview, TopRiskFactors are the three display fields.
GET /api/ml/incident-risk
Returns all rows, ordered by FlaggedForReview DESC then RiskTier.
GET /api/ml/model-meta
Returns the MlModelMeta row. Parse IncidentRiskFactors for the global risk_factors array.

In [22]:
# ── Serialize models ──────────────────────────────────────────────────────────
joblib.dump(pipe_lr, OUTPUT_DIR / "incident_risk_lr.joblib")
joblib.dump(pipe_rf, OUTPUT_DIR / "incident_risk_rf.joblib")
print("✓ Models serialized:")
print(f"  {OUTPUT_DIR / 'incident_risk_lr.joblib'}")
print(f"  {OUTPUT_DIR / 'incident_risk_rf.joblib'}")

✓ Models serialized:
  model_outputs_incident/incident_risk_lr.joblib
  model_outputs_incident/incident_risk_rf.joblib


In [23]:
# ── Build deployment JSON ─────────────────────────────────────────────────────
def json_safe(obj):
    if hasattr(obj, "item"):
        return obj.item()
    if isinstance(obj, float) and obj != obj:
        return None
    return str(obj)

# Per-resident risk output for API (Random Forest only)
RISK_FACTOR_LABELS = {
    "prior_any_incident_count": "History of prior incidents",
    "risk_level_delta": "Change in risk level since intake",
    "avg_emotional_state": "Emotional state during sessions",
    "days_since_last_incident": "Time since last incident",
    "initial_risk_level": "Risk level at intake",
    "pct_distressed_sessions": "Frequency of distressed sessions",
    "concerns_flagged_rate": "Rate of concerns flagged by counselors",
    "emotional_improvement_rate": "Emotional improvement across sessions",
    "avg_health_score": "Overall health score",
    "length_of_stay_days": "Length of stay",
    "progress_noted_rate": "Rate of progress noted in sessions",
    "session_count": "Number of counseling sessions",
}


def _risk_tier(rf_score: float) -> str:
    if rf_score >= 0.7:
        return "High Alert"
    if rf_score >= 0.4:
        return "Monitor Closely"
    return "Stable"


def _pretty_feature(name: str) -> str:
    if name in RISK_FACTOR_LABELS:
        return RISK_FACTOR_LABELS[name]
    return str(name).replace("_", " ").strip().title()


resident_scores = []
feature_names = X.columns.tolist()
for _, row in feature_df.iterrows():
    rid = int(row["resident_id"])
    rf_score = float(pipe_rf.predict_proba(pd.DataFrame([row[feature_names]]))[0, 1])

    # Top contributing features for this resident (RF importances × feature value deviation)
    feat_vals = row[feature_names]
    feat_means = X.mean()
    feat_stds = X.std().replace(0, 1)
    contributions = ((feat_vals - feat_means) / feat_stds * rf_fi).abs().sort_values(ascending=False)
    top_features = contributions.head(3).index.tolist()

    resident_scores.append(
        {
            "resident_id": rid,
            "risk_tier": _risk_tier(rf_score),
            "flagged_for_review": bool(rf_score >= 0.3),
            "top_risk_factors": [_pretty_feature(f) for f in top_features],
        }
    )

# ── Human-friendly risk factor importances (from LR coefficients) ────────────
FEATURE_LABELS = {
    "prior_any_incident_count": "History of prior incidents",
    "risk_level_delta": "Change in risk level since intake",
    "avg_emotional_state": "Emotional state during sessions",
    "days_since_last_incident": "Time since last incident",
    "initial_risk_level": "Risk level at intake",
}

# Use only mapped features; compute importance from absolute coefficients
coef_map = (
    lr_coef_df.set_index("feature")["coefficient"].to_dict()
    if "feature" in lr_coef_df.columns and "coefficient" in lr_coef_df.columns
    else {}
)
selected = {f: float(coef_map.get(f, 0.0)) for f in FEATURE_LABELS.keys() if f in coef_map}

risk_factors = []
abs_sum = sum(abs(v) for v in selected.values())
if abs_sum > 0:
    raw = [
        {
            "feature": f,
            "label": FEATURE_LABELS[f],
            "coef": float(selected[f]),
        }
        for f in selected
    ]
    # compute rounded percentages
    for r in raw:
        r["importance"] = int(round(abs(r["coef"]) / abs_sum * 100))

    # sort and fix rounding remainder so total == 100
    raw = sorted(raw, key=lambda x: x["importance"], reverse=True)
    remainder = 100 - sum(r["importance"] for r in raw)
    if raw:
        raw[0]["importance"] += remainder

    risk_factors = [{"label": r["label"], "importance": int(r["importance"])} for r in raw]

payload = {
    "meta": {
        "pipeline": "high_severity_incident_prediction",
        "n_training": int(len(y)),
        "n_positives": int(y.sum()),
        "lr_threshold": LR_THRESHOLD,
        "rf_threshold": RF_THRESHOLD,
        "lr_recall": round(lr_metrics["recall"], 3),
        "lr_precision": round(lr_metrics["precision"], 3),
        "rf_recall": round(rf_metrics["recall"], 3),
        "rf_precision": round(rf_metrics["precision"], 3),
        "auc_lr": round(roc_auc_score(y, probas_lr), 3),
        "auc_rf": round(roc_auc_score(y, probas_rf), 3),
        "sample_size_warning": f"Model trained on n={len(y)} residents. Retrain when 120+ residents served.",
        "label_definition": "Resident has ever had a High-severity incident (lifetime prevalence)",
        "primary_metric": "Recall — missing a high-risk resident is worse than a false flag",
    },
    "risk_factors": risk_factors,
    "resident_scores": resident_scores,
}

json_path = OUTPUT_DIR / "incident_risk_scores.json"
json_path.write_text(json.dumps(payload, indent=2, default=json_safe))

print(f"✓ Deployment JSON saved: {json_path}")
print(f"  Size: {json_path.stat().st_size:,} bytes")

✓ Deployment JSON saved: model_outputs_incident/incident_risk_scores.json
  Size: 16,267 bytes


In [24]:
# ── Simulated API response for one resident ───────────────────────────────────
example = next((r for r in resident_scores if r['flagged_for_review']), resident_scores[0])
print('SIMULATED API RESPONSE — GET /api/case-management/incident-risk/{resident_id}')
print(json.dumps(example, indent=2, default=json_safe))
print()
print('Frontend uses this to:')
print('  1. Show risk flag icon on staff dashboard next to this resident')
print('  2. Display top_risk_factors in the flag tooltip/modal')
print('  3. Populate Risk Alert column in caseload view (sortable by risk_tier)')

SIMULATED API RESPONSE — GET /api/case-management/incident-risk/{resident_id}
{
  "resident_id": 1,
  "risk_tier": "Monitor Closely",
  "flagged_for_review": true,
  "top_risk_factors": [
    "History of prior incidents",
    "Risk level at intake",
    "Time since last incident"
  ]
}

Frontend uses this to:
  1. Show risk flag icon on staff dashboard next to this resident
  2. Display top_risk_factors in the flag tooltip/modal
  3. Populate Risk Alert column in caseload view (sortable by risk_tier)


In [25]:
# ── Final model summary ───────────────────────────────────────────────────────
summary = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression",
            "Type": "Causal/Explanatory",
            "Threshold": LR_THRESHOLD,
            "Recall": lr_metrics["recall"],
            "Precision": lr_metrics["precision"],
            "F1": lr_metrics["f1"],
            "AUC": round(roc_auc_score(y, probas_lr), 3),
            "TP": lr_metrics["tp"],
            "FP": lr_metrics["fp"],
            "FN": lr_metrics["fn"],
        },
        {
            "Model": "Random Forest",
            "Type": "Predictive",
            "Threshold": RF_THRESHOLD,
            "Recall": rf_metrics["recall"],
            "Precision": rf_metrics["precision"],
            "F1": rf_metrics["f1"],
            "AUC": round(roc_auc_score(y, probas_rf), 3),
            "TP": rf_metrics["tp"],
            "FP": rf_metrics["fp"],
            "FN": rf_metrics["fn"],
        },
    ]
).set_index("Model")

print("FINAL MODEL SUMMARY")
print(summary.to_string())
print()
print("Files produced:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size:,} bytes)")

FINAL MODEL SUMMARY
                                   Type  Threshold  Recall  Precision        F1    AUC  TP  FP  FN
Model                                                                                             
Logistic Regression  Causal/Explanatory        0.3    0.95    0.59375  0.730769  0.816  19  13   1
Random Forest                Predictive        0.3    0.95    0.50000  0.655172  0.769  19  19   1

Files produced:
  model_outputs_incident/correlation_heatmap.png  (169,659 bytes)
  model_outputs_incident/feature_distributions.png  (51,868 bytes)
  model_outputs_incident/incident_risk_lr.joblib  (3,089 bytes)
  model_outputs_incident/incident_risk_rf.joblib  (419,394 bytes)
  model_outputs_incident/incident_risk_scores.json  (16,267 bytes)
  model_outputs_incident/model_diagnostics.png  (161,237 bytes)


---
## Web App Integration/Additional Notes

### Web App Integration
- **Staff Dashboard**: Risk flag icon next to each girl's name if `rf_flagged=true`. Clicking opens a modal listing `top_risk_factors` in plain English.
- **Caseload View**: Sortable "Risk Alert" column showing `rf_risk_score` as a percentage bar. Jennifer sorts descending to see highest-risk residents first.
- **Confidence display**: Show `lr_precision` and `lr_recall` in a small info tooltip so staff understand the model's limitations.

### Where Results Appear in the Application

**Per-resident (from `ResidentIncidentRisk`):**
- **Caseload Detail Panel**: Risk tier badge (🔴 High Alert / 🟡 Monitor Closely / 🟢 Stable) and a bulleted list of the three `TopRiskFactors` for that specific resident.
- **Caseload List**: Sortable "Risk" column with colored dot per resident. Residents where `FlaggedForReview = true` show a flag icon in the row.

**Program-level (from `MlModelMeta.IncidentRiskFactors`):**
- **Reports & Analytics Page**: Horizontal bar chart of the four risk factors with their importance percentages (51%, 35%, 8%, 6%). Titled "What Drives High-Severity Incidents." Sample size warning shown below the chart.

### Important Framing Note

Do NOT describe this model as predicting future incidents. The label is **lifetime prevalence** — whether a resident has ever had a high-severity incident. Frame in the UI as: "Based on this resident's history, the model has assessed her risk level as [RiskTier]."

### Retraining Trigger

Retrain when total resident count exceeds 120. At that point:
- Switch label to rolling 30-day forward window
- Add SMOTE oversampling for the minority class
- Increase `K_FEATURES` to 8
- Consider XGBoost as an additional model

### Integration Code Location

- GitHub Actions workflow: `.github/workflows/ml_pipeline.yml`
- DB write script: `write_partner_outputs_to_db.py`
- ASP.NET model classes: `ResidentIncidentRisk.cs`, `MlModelMeta.cs` in the Havyn web application repo

### Retraining Trigger
Retrain when total resident count exceeds 120. At that point:
- Switch label to rolling-window (30-day forward prediction) — the imbalance becomes tractable
- Increase `K_FEATURES` to 8
- Add SMOTE oversampling for the minority class
- Consider XGBoost as an additional model